# Sokoban + AI

**Mục tiêu:** xây dựng game Sokoban chạy ngay trong notebook, sau đó cho một AI "đăng nhập" vào game và tự chơi bằng các thuật toán tìm kiếm.

## Luật chơi
- Người chơi (hình tròn xanh dương) di chuyển 4 hướng **Lên / Xuống / Trái / Phải** trên các ô sàn.
- Người chơi chỉ **đẩy** được thùng (ô vuông màu cam), không kéo. Mỗi lần chỉ đẩy được **một** thùng.
- Thùng không thể đi xuyên tường hoặc đè lên thùng khác.
- **Thắng** khi mọi thùng đều nằm trên ô đích (chấm đỏ). Thùng nằm trên đích sẽ chuyển sang màu xanh lá.

## Mô hình hóa thành bài toán tìm kiếm
| Thành phần | Định nghĩa |
|---|---|
| **State** | `(vị trí người chơi, tập vị trí các thùng)`. Tường và đích là phần tĩnh, không nằm trong state |
| **Action** | `U`, `D`, `L`, `R` |
| **Transition** | Đi vào ô trống, hoặc đẩy thùng nếu ô phía sau thùng trống |
| **Goal test** | `tập thùng == tập đích` |

## Mục lục
| Mục | Nội dung | Trạng thái |
|---|---|---|
| §1 | Cài đặt & import | Xong |
| §2 | Level (định dạng XSB) | Xong |
| §3 | Game engine | Xong |
| §4 | Hiển thị: vẽ bàn chơi, phát lại, xuất GIF | Xong |
| §5 | Người chơi thật (nút bấm) | Xong |
| §6 | Giao thức đăng nhập: `GameServer` và `Agent` | Xong |
| §7 | Thuật toán tìm kiếm: BFS, DFS, UCS, A* | Xong |
| §8 | AI agent: `SearchAgent`, `RandomAgent` | Xong |
| §9 | Demo: AI đăng nhập và tự chơi | Xong |
| §10 | Thực nghiệm và so sánh | Xong |

## §1. Cài đặt & import

Cài thư viện (chạy một lần trong terminal, đã kích hoạt `.venv`):
```bash
pip install -r requirements.txt
```

In [ ]:
from __future__ import annotations

import heapq
import itertools
import random
import time
from collections import deque
from dataclasses import dataclass
from functools import lru_cache
from io import BytesIO
from pathlib import Path
from typing import NamedTuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, clear_output, display
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Circle, FancyBboxPatch, Rectangle

%matplotlib inline

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## §2. Level

Level được viết theo **định dạng XSB**, định dạng chuẩn của cộng đồng Sokoban. Nhờ vậy có thể chép thêm level từ các bộ có sẵn như Microban.

Gồm 2 level tập chơi, 5 level đầu của bộ **Microban** (David W. Skinner), và 2 level tự thiết kế khó hơn (`medium_1`, `hard_1`) để phần thực nghiệm phân biệt rõ các thuật toán.

| Ký tự | Ý nghĩa |
|:---:|---|
| `#` | Tường |
| ` ` | Sàn |
| `@` | Người chơi |
| `+` | Người chơi đứng trên đích |
| `$` | Thùng |
| `*` | Thùng nằm trên đích |
| `.` | Đích |

In [ ]:
LEVEL_TEXTS = {
    "tutorial_1": """
#######
#@ $ .#
#######
""",
    "tutorial_2": """
#######
#.   .#
# $$  #
#  @  #
#######
""",
    "microban_1": """
####
# .#
#  ###
#*@  #
#  $ #
#  ###
####
""",
    "microban_2": """
######
#    #
# #@ #
# $* #
# .* #
#    #
######
""",
    "microban_3": """
  ####
###  ####
#     $ #
# #  #$ #
# . .#@ #
#########
""",
    "microban_4": """
########
#      #
# .**$@#
#      #
#####  #
    ####
""",
    "microban_5": """
 #######
 #     #
 # .$. #
## $@$ #
#  .$. #
#      #
########
""",
    # Hai level tự thiết kế, khó hơn, để thấy rõ khác biệt giữa các thuật toán ở §10
    "medium_1": """
#########
#   #   #
# $ $ $ #
#.. @ ..#
# $   $ #
#  .#   #
#########
""",
    "hard_1": """
#########
#.  #  .#
# $ $ $ #
##  @  ##
# $ $ $ #
#. .#. .#
#########
""",
}

## §3. Game engine

Engine tách thành 2 lớp để sau này AI dùng lại được:

- **`Level`**: phần **tĩnh** của màn chơi (tường, đích, kích thước) và các hàm **thuần** (không đổi dữ liệu): `try_move(state, action)`, `is_solved(state)`. Thuật toán tìm kiếm ở §7 sẽ dùng trực tiếp các hàm này để sinh state con.
- **`SokobanGame`**: một **ván chơi** có trạng thái thay đổi: `step()`, `undo()`, `reset()`, đếm số bước/số lần đẩy, và `observe()` trả về dữ liệu dạng `dict` để gửi cho AI.

Tọa độ dùng dạng `(hàng, cột)`, gốc ở góc trên bên trái.

In [ ]:
Pos = tuple[int, int]  # (row, col)

ACTIONS: dict[str, Pos] = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}
ACTION_NAMES = {"U": "Lên", "D": "Xuống", "L": "Trái", "R": "Phải"}


class State(NamedTuple):
    """Phần động của màn chơi, hashable để dùng làm key trong tập visited."""
    player: Pos
    boxes: frozenset[Pos]


class Level:
    """Phần tĩnh của màn chơi: tường, đích, sàn. Không thay đổi trong suốt ván."""

    def __init__(self, text: str, name: str = "custom"):
        self.name = name
        rows = [line.rstrip() for line in text.strip("\n").splitlines()]
        walls, goals, boxes, player = set(), set(), set(), None
        for r, line in enumerate(rows):
            for c, ch in enumerate(line):
                if ch == "#":
                    walls.add((r, c))
                if ch in ".+*":
                    goals.add((r, c))
                if ch in "$*":
                    boxes.add((r, c))
                if ch in "@+":
                    if player is not None:
                        raise ValueError(f"[{name}] Có nhiều hơn một người chơi")
                    player = (r, c)
                if ch not in "#.+*$@ ":
                    raise ValueError(f"[{name}] Ký tự lạ {ch!r} tại {(r, c)}")
        if player is None:
            raise ValueError(f"[{name}] Không có người chơi (@ hoặc +)")
        if not boxes or len(boxes) != len(goals):
            raise ValueError(f"[{name}] Số thùng ({len(boxes)}) phải bằng số đích ({len(goals)}) và > 0")

        self.height = len(rows)
        self.width = max(len(line) for line in rows)
        self.walls = frozenset(walls)
        self.goals = frozenset(goals)
        self.floor = self._flood_fill(player)  # các ô bên trong tường, người chơi có thể tới
        if not boxes <= self.floor:
            raise ValueError(f"[{name}] Có thùng nằm ngoài khu vực chơi")
        self.initial_state = State(player, frozenset(boxes))

    def _flood_fill(self, start: Pos) -> frozenset[Pos]:
        seen, stack = {start}, [start]
        while stack:
            r, c = stack.pop()
            for dr, dc in ACTIONS.values():
                nxt = (r + dr, c + dc)
                if nxt not in seen and nxt not in self.walls \
                        and 0 <= nxt[0] < self.height and 0 <= nxt[1] < self.width:
                    seen.add(nxt)
                    stack.append(nxt)
        return frozenset(seen)

    # ---------- Hàm thuần: AI sẽ dùng lại ----------
    def try_move(self, state: State, action: str) -> tuple[State, bool] | None:
        """Trả về (state mới, có đẩy thùng không), hoặc None nếu nước đi không hợp lệ."""
        dr, dc = ACTIONS[action]
        pr, pc = state.player
        nxt = (pr + dr, pc + dc)
        if nxt in self.walls:
            return None
        if nxt in state.boxes:
            behind = (nxt[0] + dr, nxt[1] + dc)
            if behind in self.walls or behind in state.boxes:
                return None
            return State(nxt, (state.boxes - {nxt}) | {behind}), True
        return State(nxt, state.boxes), False

    def is_solved(self, state: State) -> bool:
        return state.boxes == self.goals

    def simulate(self, actions: str, start: State | None = None) -> list[State]:
        """Chạy một chuỗi nước đi, trả về danh sách state (gồm cả state đầu)."""
        states = [start or self.initial_state]
        for i, a in enumerate(actions.upper()):
            result = self.try_move(states[-1], a)
            if result is None:
                raise ValueError(f"Nước đi thứ {i + 1} ({a}) không hợp lệ")
            states.append(result[0])
        return states

    def to_text(self, state: State | None = None) -> str:
        """Chuyển state về lại chuỗi XSB (dùng để in ra hoặc gửi cho AI)."""
        state = state or self.initial_state
        lines = []
        for r in range(self.height):
            row = []
            for c in range(self.width):
                p = (r, c)
                if p in self.walls:
                    ch = "#"
                elif p in state.boxes:
                    ch = "*" if p in self.goals else "$"
                elif p == state.player:
                    ch = "+" if p in self.goals else "@"
                else:
                    ch = "." if p in self.goals else " "
                row.append(ch)
            lines.append("".join(row).rstrip())
        return "\n".join(lines)

    def __repr__(self):
        return f"Level({self.name!r}, {self.width}x{self.height}, {len(self.goals)} thùng)"


class SokobanGame:
    """Một ván chơi: giữ state hiện tại, lịch sử để undo, và thống kê."""

    def __init__(self, level: Level):
        self.level = level
        self.reset()

    def reset(self):
        self.state = self.level.initial_state
        self.history: list[tuple[State, str, bool]] = []  # (state trước đó, action, có đẩy)

    def step(self, action: str) -> bool:
        """Thực hiện một nước đi. Trả về False nếu nước đi không hợp lệ (state giữ nguyên)."""
        action = action.upper()
        if action not in ACTIONS:
            raise ValueError(f"Action phải là một trong {list(ACTIONS)}, nhận được {action!r}")
        result = self.level.try_move(self.state, action)
        if result is None:
            return False
        self.history.append((self.state, action, result[1]))
        self.state = result[0]
        return True

    def play(self, actions: str) -> int:
        """Chạy một chuỗi lệnh như 'RRUL', bỏ qua nước đi không hợp lệ. Trả về số nước đi thành công."""
        return sum(self.step(a) for a in actions if not a.isspace())

    def undo(self) -> bool:
        if not self.history:
            return False
        self.state = self.history.pop()[0]
        return True

    def is_solved(self) -> bool:
        return self.level.is_solved(self.state)

    @property
    def moves(self) -> int:
        return len(self.history)

    @property
    def pushes(self) -> int:
        return sum(pushed for _, _, pushed in self.history)

    @property
    def lurd(self) -> str:
        """Lời giải theo chuẩn LURD: chữ thường = đi, chữ HOA = đẩy thùng."""
        return "".join(a if pushed else a.lower() for _, a, pushed in self.history)

    def observe(self) -> dict:
        """Những gì gửi cho người chơi/AI: chỉ dữ liệu thuần, không lộ object bên trong."""
        return {
            "level": self.level.name,
            "map": self.level.to_text(self.state),
            "player": self.state.player,
            "boxes": sorted(self.state.boxes),
            "goals": sorted(self.level.goals),
            "moves": self.moves,
            "pushes": self.pushes,
            "solved": self.is_solved(),
        }


LEVELS = {name: Level(text, name) for name, text in LEVEL_TEXTS.items()}
for lv in LEVELS.values():
    print(lv)

### Kiểm thử engine
Kiểm tra nhanh các luật quan trọng. Cell này chạy không báo lỗi tức là engine đúng luật.

In [ ]:
test = Level("""
#######
#@$$ .#
#    .#
#######
""", "test")
g = SokobanGame(test)
start = g.state

assert not g.step("U"), "Không được đi xuyên tường"
assert not g.step("R"), "Không được đẩy 2 thùng cùng lúc"
assert g.state == start and g.moves == 0, "Nước đi sai không được làm đổi state"

assert g.play("DR") == 2
assert not g.step("U"), "Đẩy thùng vào tường phải bị chặn"  # thùng ở (1,2), phía sau là tường
assert g.state.player == (2, 2)

assert g.undo() and g.undo() and g.state == start, "Undo phải trả về đúng state cũ"

# Thắng tutorial_1 bằng "RRR": bước đầu chỉ đi, 2 bước sau đẩy thùng
g = SokobanGame(LEVELS["tutorial_1"])
g.play("RRR")
assert g.is_solved() and g.moves == 3 and g.pushes == 2 and g.lurd == "rRR"

# to_text → Level phải ra lại đúng màn chơi
for lv in LEVELS.values():
    again = Level(lv.to_text(), lv.name)
    assert again.walls == lv.walls and again.goals == lv.goals and again.initial_state == lv.initial_state

print("Engine qua hết các bài kiểm thử")
print(g.observe())

## §4. Hiển thị

- `draw_board()`: vẽ một state bằng matplotlib.
- `board_png()`: vẽ bàn chơi ra ảnh PNG, dùng cho widget ở §5 và màn hình theo dõi AI ở §6.
- `show_levels()`: xem nhanh tất cả level.
- `replay()`: phát lại một chuỗi nước đi ngay trong output của cell.
- `save_gif()`: xuất chuỗi nước đi thành ảnh GIF và nhúng vào notebook. Notebook nộp lên vẫn xem được cả quá trình, không cần chạy lại.

In [ ]:
COLORS = {
    "floor": "#f2ece4", "floor_edge": "#e2d8cc",
    "wall": "#4a4e69", "wall_edge": "#2b2d42",
    "goal": "#d1495b",
    "box": "#c9832e", "box_edge": "#7a4a14",
    "box_done": "#1baf7a", "box_done_edge": "#0f6b4a",
    "player": "#2a78d6", "player_edge": "#184f95",
}
CELL_INCH = 0.5  # kích thước mỗi ô khi vẽ


def draw_board(level: Level, state: State | None = None, ax=None, title: str | None = None,
               highlight=()):
    """Vẽ bàn chơi lên ax (tạo mới nếu chưa có). `highlight`: các ô sàn cần tô đỏ nhạt. Trả về ax."""
    state = state or level.initial_state
    if ax is None:
        _, ax = plt.subplots(figsize=(level.width * CELL_INCH, level.height * CELL_INCH + 0.4))
    ax.clear()
    for r, c in level.floor:
        ax.add_patch(Rectangle((c, r), 1, 1, fc=COLORS["floor"], ec=COLORS["floor_edge"], lw=0.6))
    for r, c in highlight:
        ax.add_patch(Rectangle((c, r), 1, 1, fc=COLORS["goal"], alpha=0.25, lw=0))
    for r, c in level.walls:
        ax.add_patch(Rectangle((c, r), 1, 1, fc=COLORS["wall"], ec=COLORS["wall_edge"], lw=1))
    for r, c in level.goals:
        ax.add_patch(Circle((c + 0.5, r + 0.5), 0.16, fc=COLORS["goal"]))
    for r, c in state.boxes:
        done = (r, c) in level.goals
        ax.add_patch(FancyBboxPatch(
            (c + 0.14, r + 0.14), 0.72, 0.72, boxstyle="round,pad=0,rounding_size=0.12",
            fc=COLORS["box_done" if done else "box"], ec=COLORS["box_done_edge" if done else "box_edge"], lw=1.5))
    pr, pc = state.player
    ax.add_patch(Circle((pc + 0.5, pr + 0.5), 0.3, fc=COLORS["player"], ec=COLORS["player_edge"], lw=1.5))
    ax.set_xlim(0, level.width)
    ax.set_ylim(level.height, 0)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)
    return ax


def board_png(level: Level, state: State, title: str | None = None) -> bytes:
    """Vẽ bàn chơi ra ảnh PNG (bytes) để cập nhật widget/ảnh tại chỗ mà không xóa output khác."""
    fig, ax = plt.subplots(figsize=(level.width * CELL_INCH, level.height * CELL_INCH + 0.4))
    draw_board(level, state, ax=ax, title=title)
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


def show_board(level: Level, state: State | None = None, title: str | None = None):
    draw_board(level, state, title=title or level.name)
    plt.show()


def show_levels(levels: dict[str, Level] = LEVELS, ncols: int = 4):
    n = len(levels)
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
    axes = axes.ravel() if n > 1 else [axes]
    for ax, lv in zip(axes, levels.values()):
        draw_board(lv, ax=ax, title=f"{lv.name}  ({len(lv.goals)} thùng)")
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


def _frame_title(level: Level, actions: str, i: int, pushes: int) -> str:
    last = f"  •  {ACTION_NAMES[actions[i - 1]]}" if i else ""
    return f"{level.name} — bước {i}/{len(actions)}  •  đẩy {pushes}{last}"


def replay(level: Level, actions: str, delay: float = 0.15):
    """Phát lại chuỗi nước đi ngay trong output (chỉ khung hình cuối được lưu vào file .ipynb)."""
    actions = actions.upper()
    states = level.simulate(actions)
    pushes = 0
    for i, s in enumerate(states):
        if i and s.boxes != states[i - 1].boxes:
            pushes += 1
        clear_output(wait=True)
        show_board(level, s, title=_frame_title(level, actions, i, pushes))
        time.sleep(delay)


def save_gif(level: Level, actions: str, filename: str | None = None, fps: int = 6) -> Image:
    """Xuất chuỗi nước đi thành GIF trong outputs/ và trả về ảnh để hiển thị (được nhúng vào notebook)."""
    actions = actions.upper()
    states = level.simulate(actions)
    pushes = [0]
    for prev, cur in zip(states, states[1:]):
        pushes.append(pushes[-1] + (cur.boxes != prev.boxes))
    fig, ax = plt.subplots(figsize=(level.width * CELL_INCH, level.height * CELL_INCH + 0.4))

    def update(i):
        draw_board(level, states[i], ax=ax, title=_frame_title(level, actions, i, pushes[i]))

    # Giữ khung hình cuối lâu hơn để người xem kịp thấy kết quả
    frames = list(range(len(states))) + [len(states) - 1] * fps
    anim = FuncAnimation(fig, update, frames=frames)
    path = OUTPUT_DIR / (filename or f"{level.name}.gif")
    anim.save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return Image(filename=str(path))

In [ ]:
show_levels()

### Ví dụ: in level dạng chữ và phát lại một lời giải viết tay

In [ ]:
print(LEVELS["tutorial_2"].to_text())

In [ ]:
# Lời giải viết tay cho tutorial_2: đẩy thùng trái lên rồi sang trái, thùng phải sang phải rồi lên
HAND_SOLUTION = "LUDRRUULLDRRDRU"
game = SokobanGame(LEVELS["tutorial_2"])
game.play(HAND_SOLUTION)
print(f"Đã giải: {game.is_solved()}  |  {game.moves} bước, {game.pushes} lần đẩy  |  LURD: {game.lurd}")

In [ ]:
replay(LEVELS["tutorial_2"], HAND_SOLUTION, delay=0.2)

In [ ]:
save_gif(LEVELS["tutorial_2"], HAND_SOLUTION)

## §5. Người chơi thật

Bấm các nút mũi tên để chơi. Ngoài ra có thể gõ một **chuỗi lệnh** (vd: `RRUL`) rồi nhấn **Enter**.

> **Lưu ý:** Widget chỉ tương tác được khi notebook đang chạy kernel. Mở file đã lưu mà không chạy kernel thì sẽ không thấy bàn chơi này.

In [ ]:
def play(level_name: str = "tutorial_1"):
    game = SokobanGame(LEVELS[level_name])
    # Dùng widgets.Image thay vì widgets.Output: Output thuộc module @jupyter-widgets/output,
    # module này không có trong bộ widget của VS Code nên sẽ báo lỗi "No version ... is registered".
    board = widgets.Image(format="png")
    status = widgets.HTML()

    def refresh(message: str = ""):
        board.value = board_png(game.level, game.state, game.level.name)
        solved = "<b>Hoàn thành!</b>" if game.is_solved() else ""
        status.value = (f"Bước: <b>{game.moves}</b> &nbsp; Đẩy: <b>{game.pushes}</b> &nbsp; "
                        f"LURD: <code>{game.lurd or '-'}</code> &nbsp; {solved} {message}")

    def move(action: str):
        if game.is_solved():
            return refresh("— bấm Reset để chơi lại")
        refresh("" if game.step(action) else "Không đi được")

    def button(label: str, on_click, width: str = "48px"):
        b = widgets.Button(description=label, layout=widgets.Layout(width=width))
        b.on_click(lambda _: on_click())
        return b

    spacer = widgets.Box(layout=widgets.Layout(width="52px"))
    pad = widgets.VBox([
        widgets.HBox([spacer, button("↑", lambda: move("U"))]),
        widgets.HBox([button("←", lambda: move("L")), button("↓", lambda: move("D")), button("→", lambda: move("R"))]),
    ])

    def undo():
        refresh("" if game.undo() else "Không còn gì để undo")

    def reset():
        game.reset()
        refresh()

    level_picker = widgets.Dropdown(options=list(LEVELS), value=level_name, description="Level:")

    def change_level(change):
        nonlocal game
        game = SokobanGame(LEVELS[change["new"]])
        refresh()

    level_picker.observe(change_level, names="value")

    command = widgets.Text(placeholder="Chuỗi lệnh, vd: RRUL", continuous_update=False,
                           layout=widgets.Layout(width="200px"))

    def run_command(change):
        text = change["new"].strip().upper()
        if not text:
            return
        bad = set(text) - set(ACTIONS) - {" "}
        if bad:
            refresh(f"Ký tự không hợp lệ: {''.join(sorted(bad))}")
        else:
            done = game.play(text)
            skipped = len(text.replace(" ", "")) - done
            refresh(f"(bỏ qua {skipped} nước không hợp lệ)" if skipped else "")
        command.value = ""

    command.observe(run_command, names="value")

    controls = widgets.HBox([pad, widgets.VBox([
        widgets.HBox([button("Undo", undo, "70px"), button("Reset", reset, "70px")]),
        command,
    ])], layout=widgets.Layout(gap="24px", align_items="center"))
    display(widgets.VBox([level_picker, board, status, controls]))
    refresh()

In [ ]:
play("microban_1")

## §6. Giao thức "đăng nhập" cho AI

Game và AI giao tiếp giống một server với một client:

```
GameServer                                   Agent
    |  connect(agent) ── welcome ─────────>  login()   → trả về tên
    |                                           |
    |  ── observation (dict) ─────────────>  act()     → "U" / "D" / "L" / "R" / None
    |  game.step(action), vẽ lại bàn chơi       |
    |  ... lặp cho tới khi thắng / agent bỏ cuộc / hết lượt
    |  ── result ──────────────────────────>  on_game_over()
```

- Agent **chỉ nhận dữ liệu thuần** (`game.observe()`: bản đồ dạng chữ XSB, vị trí người và thùng...). Agent không được truy cập object `SokobanGame` bên trong server. Vì vậy có thể thay AI khác mà không phải sửa game.
- Action không hợp lệ (đi vào tường, sai ký tự) bị bỏ qua nhưng **vẫn tính vào số lượt**. Nhờ vậy agent viết sai không thể làm vòng lặp chạy mãi.

In [ ]:
class Agent:
    """Lớp cơ sở cho mọi người chơi tự động."""
    name = "Agent"

    def login(self, welcome: dict) -> str:
        """Server gọi khi agent kết nối. Trả về tên hiển thị."""
        return self.name

    def act(self, observation: dict) -> str | None:
        """Nhận trạng thái hiện tại, trả về một action trong ACTIONS, hoặc None để bỏ cuộc."""
        raise NotImplementedError

    def on_game_over(self, result: dict) -> None:
        """Server gọi khi ván kết thúc."""


class GameServer:
    """Giữ ván chơi, cho một agent đăng nhập, rồi lặp: gửi observation → nhận action → cập nhật game."""

    def __init__(self, level: Level, max_steps: int = 2000):
        self.level = level
        self.max_steps = max_steps
        self.game = SokobanGame(level)
        self.agent: Agent | None = None
        self.player_name: str | None = None

    def connect(self, agent: Agent) -> str:
        welcome = {"level": self.level.name, "actions": list(ACTIONS), "max_steps": self.max_steps}
        self.player_name = agent.login(welcome)
        self.agent = agent
        self.game.reset()
        print(f"[server] {self.player_name} đã đăng nhập vào level '{self.level.name}'")
        return self.player_name

    def _title(self) -> str:
        return f"{self.player_name} — {self.level.name} — bước {self.game.moves} • đẩy {self.game.pushes}"

    def run(self, delay: float = 0.15, render: bool = True) -> dict:
        """Chạy ván chơi. render=True: vẽ lại bàn chơi tại chỗ sau mỗi nước đi."""
        if self.agent is None:
            raise RuntimeError("Chưa có agent nào đăng nhập. Gọi connect(agent) trước.")
        game = self.game
        screen = display(Image(board_png(self.level, game.state, self._title())), display_id=True) if render else None
        invalid, think_time, gave_up = 0, 0.0, False

        while not game.is_solved() and game.moves + invalid < self.max_steps:
            observation = game.observe()
            t0 = time.perf_counter()
            action = self.agent.act(observation)
            think_time += time.perf_counter() - t0
            if action is None:
                gave_up = True
                break
            if action not in ACTIONS or not game.step(action):
                invalid += 1
                continue
            if screen:
                screen.update(Image(board_png(self.level, game.state, self._title())))
                time.sleep(delay)

        result = {
            "player": self.player_name,
            "level": self.level.name,
            "solved": game.is_solved(),
            "moves": game.moves,
            "pushes": game.pushes,
            "invalid_actions": invalid,
            "gave_up": gave_up,
            "think_time": round(think_time, 3),
            "lurd": game.lurd,
        }
        self.agent.on_game_over(result)
        if result["solved"]:
            verdict = "GIẢI ĐƯỢC"
        elif gave_up:
            verdict = "BỎ CUỘC"
        else:
            verdict = f"HẾT LƯỢT ({self.max_steps})"
        print(f"[server] Kết thúc: {verdict} | {game.moves} bước, {game.pushes} lần đẩy, "
              f"{invalid} nước không hợp lệ | thời gian suy nghĩ {think_time:.3f}s")
        return result

## §7. Thuật toán tìm kiếm

### Chi phí
Nếu mọi bước có chi phí bằng nhau thì UCS cho kết quả giống hệt BFS. Vì vậy ta đặt:
- Đi bộ một bước: **1**
- Đẩy thùng một bước: **`PUSH_COST = 2`** (đẩy thùng tốn sức hơn)

Như vậy BFS tìm lời giải **ít bước nhất**, còn UCS và A* tìm lời giải **ít tốn công nhất**, tức là ưu tiên ít lần đẩy.

### Các thuật toán
| Thuật toán | Frontier | Tối ưu? |
|---|---|---|
| **BFS** | Hàng đợi FIFO | Ít bước nhất |
| **DFS** | Ngăn xếp LIFO | Không, lời giải thường rất dài |
| **UCS** | Hàng đợi ưu tiên theo `g` (chi phí đã đi) | Chi phí nhỏ nhất |
| **A\*** | Hàng đợi ưu tiên theo `f = g + h` | Chi phí nhỏ nhất (vì `h` admissible) |

Cả bốn thuật toán đều là **graph search**: lưu các state đã gặp để không mở rộng lại.

### Heuristic cho A*
`h(state) = PUSH_COST × Σ khoảng cách Manhattan từ mỗi thùng tới đích gần nhất`

Mỗi lần đẩy chỉ dời một thùng đi một ô, nên cần ít nhất ngần ấy lần đẩy. Vậy `h` không bao giờ vượt quá chi phí thật, tức là **admissible**, và A* vẫn tìm được lời giải tối ưu.

### Cắt nhánh deadlock (dead squares)
Một thùng bị đẩy vào **góc tường không phải đích** thì không bao giờ ra được nữa. Ta tính trước các **ô chết** bằng cách "kéo" thùng ngược từ mỗi đích ra: ô nào kéo tới được là ô sống, còn lại là ô chết. Khi sinh state con, nước đẩy nào đưa thùng vào ô chết thì bỏ luôn. Nhờ vậy số state phải duyệt giảm đi rất nhiều.

In [ ]:
PUSH_COST = 2  # chi phí một lần đẩy thùng; đi bộ tốn 1


@lru_cache(maxsize=None)
def dead_squares(level: Level) -> frozenset[Pos]:
    """Các ô mà thùng bị đẩy vào thì không bao giờ tới được đích.
    Cách tính: từ mỗi đích 'kéo' thùng ngược lại; ô nào kéo tới được là ô sống."""
    live, stack = set(level.goals), list(level.goals)
    while stack:
        r, c = stack.pop()
        for dr, dc in ACTIONS.values():
            box, player = (r + dr, c + dc), (r + 2 * dr, c + 2 * dc)  # người đứng ở `player`, kéo thùng về `box`
            if box not in live and box in level.floor and player in level.floor:
                live.add(box)
                stack.append(box)
    return level.floor - live


def successors(level: Level, state: State, dead: frozenset[Pos]):
    """Sinh (action, state con, chi phí bước), bỏ qua nước đẩy thùng vào ô chết."""
    for action, (dr, dc) in ACTIONS.items():
        moved = level.try_move(state, action)
        if moved is None:
            continue
        nxt, pushed = moved
        if pushed and (nxt.player[0] + dr, nxt.player[1] + dc) in dead:
            continue
        yield action, nxt, PUSH_COST if pushed else 1


def heuristic(level: Level, state: State) -> int:
    """PUSH_COST × tổng khoảng cách Manhattan từ mỗi thùng tới đích gần nhất (admissible)."""
    return PUSH_COST * sum(min(abs(br - gr) + abs(bc - gc) for gr, gc in level.goals)
                           for br, bc in state.boxes)


def path_stats(level: Level, path: str) -> tuple[int, int]:
    """Trả về (số lần đẩy, tổng chi phí) của một lời giải."""
    states = level.simulate(path)
    pushes = sum(a.boxes != b.boxes for a, b in zip(states, states[1:]))
    return pushes, len(path) + (PUSH_COST - 1) * pushes

In [ ]:
class SearchLimit(Exception):
    """Dừng tìm kiếm vì hết thời gian hoặc vượt số node cho phép."""


class SearchContext:
    """Phần dùng chung cho mọi thuật toán: bảng parent, đếm node, giới hạn tài nguyên."""

    def __init__(self, level: Level, time_limit: float, max_expanded: int, prune: bool):
        self.level = level
        self.dead = dead_squares(level) if prune else frozenset()
        self.parent: dict[State, tuple[State, str] | None] = {level.initial_state: None}
        self.expanded = 0
        self.generated = 0
        self.deadline = time.perf_counter() + time_limit
        self.max_expanded = max_expanded

    def expand(self, state: State):
        """Mở rộng một node: đếm, kiểm tra giới hạn, rồi sinh các state con."""
        self.expanded += 1
        if self.expanded > self.max_expanded:
            raise SearchLimit("node_limit")
        if self.expanded % 1024 == 0 and time.perf_counter() > self.deadline:
            raise SearchLimit("timeout")
        for action, nxt, cost in successors(self.level, state, self.dead):
            self.generated += 1
            yield action, nxt, cost

    def path_to(self, goal: State) -> str:
        actions = []
        state = goal
        while self.parent[state] is not None:
            state, action = self.parent[state]
            actions.append(action)
        return "".join(reversed(actions))


def bfs(ctx: SearchContext) -> State | None:
    frontier = deque([ctx.level.initial_state])
    while frontier:
        state = frontier.popleft()
        if ctx.level.is_solved(state):
            return state
        for action, nxt, _ in ctx.expand(state):
            if nxt not in ctx.parent:
                ctx.parent[nxt] = (state, action)
                frontier.append(nxt)
    return None


def dfs(ctx: SearchContext) -> State | None:
    frontier = [ctx.level.initial_state]
    while frontier:
        state = frontier.pop()
        if ctx.level.is_solved(state):
            return state
        # Đảo thứ tự để action đầu tiên (U) được lấy ra trước
        for action, nxt, _ in reversed(list(ctx.expand(state))):
            if nxt not in ctx.parent:
                ctx.parent[nxt] = (state, action)
                frontier.append(nxt)
    return None


def best_first(ctx: SearchContext, h) -> State | None:
    """Khung chung cho UCS (h = 0) và A*. Frontier sắp theo (f = g + h, h)."""
    start = ctx.level.initial_state
    best_g = {start: 0}
    tie = itertools.count()  # tránh so sánh State khi f và h bằng nhau
    frontier = [(h(start), h(start), next(tie), 0, start)]
    while frontier:
        _, _, _, g, state = heapq.heappop(frontier)
        if g > best_g[state]:
            continue  # bản cũ trong heap, đã có đường rẻ hơn tới state này
        if ctx.level.is_solved(state):
            return state
        for action, nxt, cost in ctx.expand(state):
            new_g = g + cost
            if new_g < best_g.get(nxt, float("inf")):
                best_g[nxt] = new_g
                ctx.parent[nxt] = (state, action)
                hn = h(nxt)
                heapq.heappush(frontier, (new_g + hn, hn, next(tie), new_g, nxt))
    return None


def ucs(ctx: SearchContext) -> State | None:
    return best_first(ctx, lambda s: 0)


def astar(ctx: SearchContext) -> State | None:
    return best_first(ctx, lambda s: heuristic(ctx.level, s))


SOLVERS = {"BFS": bfs, "DFS": dfs, "UCS": ucs, "A*": astar}


@dataclass
class SearchResult:
    algorithm: str
    level: str
    status: str  # "solved" | "no_solution" | "timeout" | "node_limit"
    path: str
    pushes: int
    cost: int
    expanded: int
    generated: int
    seconds: float

    @property
    def solved(self) -> bool:
        return self.status == "solved"

    def __str__(self):
        head = f"{self.algorithm:>3} | {self.level:<11} | "
        if not self.solved:
            return head + f"{self.status.upper()} sau {self.expanded:,} node, {self.seconds:.2f}s"
        return (head + f"{len(self.path):>4} bước, {self.pushes:>3} đẩy, chi phí {self.cost:>4} | "
                f"mở rộng {self.expanded:>9,} node | {self.seconds:6.2f}s")


def solve(level: Level, algorithm: str = "A*", time_limit: float = 60,
          max_expanded: int = 3_000_000, prune: bool = True) -> SearchResult:
    ctx = SearchContext(level, time_limit, max_expanded, prune)
    t0 = time.perf_counter()
    try:
        goal = SOLVERS[algorithm](ctx)
        status = "solved" if goal else "no_solution"
    except SearchLimit as limit:
        goal, status = None, str(limit)
    seconds = time.perf_counter() - t0
    path = ctx.path_to(goal) if goal else ""
    pushes, cost = path_stats(level, path)
    return SearchResult(algorithm, level.name, status, path, pushes, cost, ctx.expanded, ctx.generated, seconds)

### Xem các ô chết
Ô tô đỏ là ô chết: đẩy thùng vào đó là thua chắc, nên thuật toán bỏ qua ngay.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, name in zip(axes, ["microban_1", "microban_3", "microban_4"]):
    lv = LEVELS[name]
    draw_board(lv, ax=ax, title=f"{name}: {len(dead_squares(lv))} ô chết", highlight=dead_squares(lv))
fig.tight_layout()
plt.show()

### Chạy thử 4 thuật toán trên một level

In [ ]:
for algo in SOLVERS:
    print(solve(LEVELS["tutorial_2"], algo))

## §8. AI agent

- **`SearchAgent`**: ở lượt đầu tiên, agent tự dựng lại `Level` từ bản đồ dạng chữ mà server gửi tới, chạy thuật toán tìm kiếm để có cả lời giải, rồi mỗi lượt sau trả về một nước đi trong lời giải đó. Nếu không tìm được lời giải thì agent trả về `None` để bỏ cuộc.
- **`RandomAgent`**: đi ngẫu nhiên, dùng làm mốc so sánh để thấy vai trò của thuật toán tìm kiếm.

In [ ]:
class SearchAgent(Agent):
    def __init__(self, algorithm: str = "A*", time_limit: float = 60):
        if algorithm not in SOLVERS:
            raise ValueError(f"algorithm phải là một trong {list(SOLVERS)}")
        self.algorithm = algorithm
        self.time_limit = time_limit
        self.name = f"SearchAgent[{algorithm}]"
        self.plan: deque[str] = deque()
        self.search: SearchResult | None = None

    def login(self, welcome: dict) -> str:
        self.plan.clear()
        self.search = None
        return self.name

    def act(self, observation: dict) -> str | None:
        if self.search is None:
            # Agent chỉ có bản đồ dạng chữ: tự dựng lại Level rồi tìm lời giải một lần
            level = Level(observation["map"], observation["level"])
            print(f"[{self.name}] Đang tìm lời giải...")
            self.search = solve(level, self.algorithm, time_limit=self.time_limit)
            print(f"[{self.name}] {self.search}")
            self.plan = deque(self.search.path)
        return self.plan.popleft() if self.plan else None


class RandomAgent(Agent):
    def __init__(self, seed: int | None = None):
        self.rng = random.Random(seed)
        self.name = "RandomAgent"

    def act(self, observation: dict) -> str:
        return self.rng.choice(list(ACTIONS))

## §9. Demo: AI đăng nhập và tự chơi

Chạy cell dưới để xem AI chơi ngay trong output. Muốn thử level hoặc thuật toán khác thì đổi `DEMO_LEVEL` và `DEMO_ALGORITHM`.

In [ ]:
DEMO_LEVEL = "microban_2"
DEMO_ALGORITHM = "A*"

server = GameServer(LEVELS[DEMO_LEVEL])
server.connect(SearchAgent(DEMO_ALGORITHM))
demo_result = server.run(delay=0.15)

Ảnh động bên trên chỉ lưu khung hình cuối khi nộp file. GIF dưới đây lưu lại toàn bộ ván chơi của AI:

In [ ]:
save_gif(LEVELS[DEMO_LEVEL], demo_result["lurd"], f"ai_{DEMO_LEVEL}.gif")

### So sánh với người chơi ngẫu nhiên
`RandomAgent` có tối đa 2000 lượt. Level nhỏ như `tutorial_2` thì đi ngẫu nhiên đôi khi cũng thắng nhờ may mắn, nhưng level thật thì gần như chắc chắn hết lượt.

In [ ]:
for name in ["tutorial_2", "microban_1", "microban_5"]:
    server = GameServer(LEVELS[name], max_steps=2000)
    server.connect(RandomAgent(seed=0))
    server.run(render=False)

## §10. Thực nghiệm và so sánh

Chạy cả 4 thuật toán trên mọi level, mỗi lần giới hạn **60 giây** và 3 triệu node.
Toàn bộ phần này chạy mất khoảng 2–4 phút tùy máy, chủ yếu do level `hard_1`. Các chỉ số ghi lại:
- `moves`: độ dài lời giải (số bước)
- `pushes`: số lần đẩy thùng
- `cost`: tổng chi phí (đi = 1, đẩy = `PUSH_COST`)
- `expanded`: số node đã mở rộng, phản ánh lượng công việc của thuật toán
- `seconds`: thời gian chạy

In [ ]:
EXPERIMENT_TIME_LIMIT = 60

rows = []
for lv in LEVELS.values():
    for algo in SOLVERS:
        r = solve(lv, algo, time_limit=EXPERIMENT_TIME_LIMIT)
        print(r)
        rows.append({
            "level": r.level, "algorithm": r.algorithm, "status": r.status,
            "moves": len(r.path) if r.solved else None, "pushes": r.pushes if r.solved else None,
            "cost": r.cost if r.solved else None, "expanded": r.expanded, "seconds": r.seconds,
        })
results = pd.DataFrame(rows)

In [ ]:
results.round({"seconds": 4})

### Bảng tóm tắt theo từng chỉ số

In [ ]:
for metric in ["moves", "cost", "expanded", "seconds"]:
    print(f"\n=== {metric} ===")
    table = results.pivot(index="level", columns="algorithm", values=metric)
    display(table.reindex(index=list(LEVELS), columns=list(SOLVERS)).round(4))

In [ ]:
ALGO_COLORS = {"BFS": "#2a78d6", "DFS": "#eb6834", "UCS": "#1baf7a", "A*": "#eda100"}
INK, MUTED, GRID = "#2b2b2b", "#6b6b6b", "#e3e3e3"


def plot_comparison(df: pd.DataFrame):
    metrics = [("seconds", "Thời gian chạy (giây)"),
               ("expanded", "Số node mở rộng"),
               ("moves", "Độ dài lời giải (bước)")]
    levels = list(dict.fromkeys(df["level"]))
    algos = list(SOLVERS)
    x = np.arange(len(levels))
    width = 0.8 / len(algos)

    fig, axes = plt.subplots(len(metrics), 1, figsize=(11, 10), sharex=True)
    for ax, (col, label) in zip(axes, metrics):
        for i, algo in enumerate(algos):
            sub = df[df["algorithm"] == algo].set_index("level").reindex(levels)
            solved = (sub["status"] == "solved").to_numpy()
            xs = x + (i - (len(algos) - 1) / 2) * width
            ax.bar(xs[solved], sub[col].to_numpy(dtype=float)[solved], width=width * 0.85,
                   color=ALGO_COLORS[algo], label=algo, zorder=2)
            for xi in xs[~solved]:  # không giải được trong giới hạn
                ax.text(xi, 0.03, "×", transform=ax.get_xaxis_transform(),
                        ha="center", va="bottom", color=MUTED, fontsize=11)
        ax.set_yscale("log")
        ax.set_title(label + " — thang log", loc="left", fontsize=11, color=INK)
        ax.grid(axis="y", color=GRID, linewidth=0.6, zorder=0)
        ax.set_axisbelow(True)
        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)
        ax.tick_params(colors=MUTED, length=0)
    axes[-1].set_xticks(x, levels, color=INK)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=len(algos), frameon=False, bbox_to_anchor=(0.5, 1.0))
    fig.text(0.99, 0.0, "× = không giải được trong giới hạn thời gian", ha="right", color=MUTED, fontsize=9)
    fig.tight_layout(rect=(0, 0.02, 1, 0.96))
    plt.show()


plot_comparison(results)

### Ảnh hưởng của việc cắt nhánh deadlock
So sánh A* có và không có cắt nhánh ô chết trên các level nhiều thùng.

In [ ]:
prune_rows = []
for name in ["microban_4", "microban_5", "medium_1"]:
    for prune in (False, True):
        r = solve(LEVELS[name], "A*", time_limit=EXPERIMENT_TIME_LIMIT, prune=prune)
        prune_rows.append({"level": name, "cắt nhánh": "có" if prune else "không", "status": r.status,
                           "cost": r.cost if r.solved else None, "expanded": r.expanded,
                           "seconds": round(r.seconds, 3)})
pd.DataFrame(prune_rows)

### Nhận xét

Số node mở rộng, độ dài lời giải và chi phí là cố định giữa các lần chạy. Thời gian chạy thì thay đổi tùy máy.

1. **Chất lượng lời giải.** BFS luôn cho lời giải **ít bước nhất**. UCS và A* luôn cho lời giải **chi phí nhỏ nhất**. Ở `microban_5`, A* đi 27 bước nhưng chỉ đẩy 6 lần, còn BFS đi 25 bước nhưng đẩy 8 lần. Cả hai cùng chi phí 33, nhưng A* đẩy ít hơn.
2. **DFS nhanh nhưng lời giải kém.** DFS thường mở rộng ít node nhất vì lao sâu theo một hướng, nhưng lời giải rất dài và vòng vèo: 6556 bước ở `medium_1` so với 32 bước của BFS. DFS chỉ hợp khi cần *một* lời giải bất kỳ thật nhanh, không quan tâm độ dài.
3. **Heuristic giúp A\* mạnh hơn hẳn trên level khó.** Ở `medium_1`, A* chỉ mở rộng khoảng 192 nghìn node, trong khi UCS mở rộng khoảng 724 nghìn, gấp gần 4 lần. Ở `hard_1`, A* là thuật toán tối ưu duy nhất giải được trong giới hạn, còn BFS và UCS đều dừng giữa chừng. Trên level nhỏ thì khác biệt không đáng kể, và A* có thể chậm hơn chút vì phải tính heuristic cho mỗi node.
4. **UCS gần giống BFS.** Chi phí chỉ khác nhau giữa 1 (đi) và 2 (đẩy), nên thứ tự duyệt của UCS gần giống BFS và số node mở rộng cũng xấp xỉ nhau.
5. **Cắt nhánh ô chết hiệu quả.** Với A*, số node mở rộng giảm khoảng 8 lần ở `microban_4`, 18 lần ở `microban_5`, và hơn 2 lần ở `medium_1`. Chi phí lời giải không đổi.
6. **Random agent.** Đi ngẫu nhiên chỉ thắng được level rất nhỏ. Từ `microban_1` trở đi là hết 2000 lượt mà không giải được, cho thấy cần đến tìm kiếm có hệ thống.

**Hướng cải tiến:**
- Heuristic tốt hơn: dùng khoảng cách đẩy thật (có tính tường) thay cho Manhattan, hoặc ghép cặp thùng–đích tối ưu.
- Phát hiện thêm loại deadlock "đóng băng", khi 2 thùng kẹt nhau sát tường.
- Tìm kiếm ở mức "lần đẩy" thay vì từng bước đi để thu nhỏ không gian trạng thái.
